In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [21]:
file = r'C:\Users\emilian\OneDrive\ITS\Esami\Esame Machine Learning\indennizzi2.csv'
dati_full = pd.read_csv(file)

In [22]:
# colonne id non servono per allenare il modello
dati_full = dati_full.drop(columns=["IdentificativoCaso", "IdentificativoLavoratore"])

In [23]:
# converto la stringa della data in un ogetto data
date = pd.to_datetime(dati_full["DataProtocollo"], dayfirst=True)
dati_full["Anno"] = date.dt.year
dati_full["Mese"] = date.dt.month
# sostituisto la stringa originale con il mese e anno
dati_full = dati_full.drop(columns=["DataProtocollo"])

In [24]:
# trasformo i dati testuali con dati numerici
col_categoriche = ["Genere", "SettoreCorrelatoMalattia", "SubSettoreCorrelatoMalattia",
                   "QualificazioneLegge", "Indennizzo"]
le = LabelEncoder()
for col in col_categoriche:
    dati_full[col] = le.fit_transform(dati_full[col])

In [25]:
# sostituisco i dati nun con la mediana di quella colonna
for col in ["GradoMenomazioneLavoratore", "GradoMenomazioneCaso"]:
    dati_full[col] = dati_full[col].fillna(dati_full[col].median())

In [26]:
X = dati_full.drop(columns=["GiorniIndennizzati"])
y = dati_full["GiorniIndennizzati"]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

In [28]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
param_grid = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [5, 10, 20, 50, None],
    'min_samples_leaf':[1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
    }

grid_search = GridSearchCV(RandomForestRegressor(), param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("media di tutte le combinazione: ", grid_search.cv_results_["mean_test_score"])
print("parametri migliori : ", grid_search.best_params_)
print("miglior risultato: ", grid_search.best_score_)

media di tutte le combinazione:  [0.61931214 0.62211282 0.62200096 0.62192251 0.61927508 0.62098176
 0.62173302 0.62223904 0.62110368 0.6198019  0.62043552 0.6204732
 0.60886048 0.61624086 0.61423807 0.61682142 0.62065389 0.62199612
 0.6222053  0.62230025 0.62204221 0.61973525 0.62246249 0.62239828
 0.61902652 0.61706834 0.62008332 0.62032047 0.61434182 0.61406243
 0.61536239 0.61550371 0.61760073 0.61950848 0.61996382 0.61864624
 0.62134922 0.62306888 0.62332055 0.62259858 0.61972212 0.62223219
 0.62286755 0.62319219 0.61995342 0.61938397 0.61757462 0.61880122
 0.63682271 0.63957991 0.63954311 0.64183986 0.63796081 0.6409078
 0.64407416 0.64362953 0.63808168 0.6377163  0.63947537 0.63976496
 0.6277275  0.62750476 0.62869081 0.6284219  0.63855099 0.63948877
 0.64262765 0.64028084 0.64048389 0.64263225 0.64299275 0.64392704
 0.63736461 0.63851003 0.63847033 0.63876466 0.63129824 0.62825064
 0.62824211 0.6286512  0.61753817 0.6221986  0.62228285 0.62148396
 0.62596179 0.62712601 0.627780

In [30]:
reg1 = RandomForestRegressor(**grid_search.best_params_, n_jobs=-1, random_state=42)
reg1.fit(X_train, y_train)
y_pred1 = reg1.predict(X_test)

In [31]:
print("R2: ", reg1.score(X_test, y_test))
print("MAE: ", mean_absolute_error(y_pred1, y_test))

R2:  0.5834909895043722
MAE:  19.14942486988467


In [32]:
X_clf = X_train
y_clf = (y_train > 0)

In [33]:
# preparo ad allenare il regressore solo con giorni di indenizzo superiori a 0
X_reg = X_train[y_clf]
y_reg = y_train[y_clf]

In [34]:
grid_search_clf = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, n_jobs=-1)
grid_search_clf.fit(X_clf, y_clf)

print("media di tutte le combinazioni: ", grid_search_clf.cv_results_["mean_test_score"])
print("parametri migliori: ", grid_search_clf.best_params_)
print("miglior risultato: ", grid_search_clf.best_score_)

media di tutte le combinazioni:  [0.9137838  0.91299015 0.91299015 0.91299015 0.91299329 0.91219964
 0.91219964 0.91219964 0.91219964 0.91140912 0.91219964 0.91219964
 0.90508187 0.90587239 0.90587239 0.9066629  0.91220277 0.91299015
 0.91299015 0.91299015 0.91061861 0.91140912 0.91219964 0.91299329
 0.91141226 0.91299329 0.91140912 0.91140912 0.90666604 0.90508187
 0.90666604 0.90508187 0.90824707 0.90666604 0.90587553 0.90666604
 0.90824707 0.90903758 0.90745655 0.90745655 0.90824707 0.90824707
 0.90824707 0.90903758 0.90113244 0.90192296 0.90192296 0.90192296
 0.92090156 0.92248886 0.92327938 0.92248573 0.92248886 0.92011105
 0.92248886 0.92248886 0.91694899 0.9193174  0.91852688 0.91773951
 0.90903758 0.9082502  0.9082502  0.90904072 0.92011419 0.92248886
 0.92169521 0.92169521 0.92328251 0.92248573 0.9209047  0.92327938
 0.91852688 0.91694586 0.91615534 0.91853002 0.90824707 0.90904072
 0.90904072 0.9082502  0.91061233 0.91298388 0.91377753 0.91536169
 0.91535855 0.91298701 0.9161

In [35]:
grid_search_reg = GridSearchCV(RandomForestRegressor(), param_grid, cv=5, n_jobs=-1)
grid_search_reg.fit(X_reg, y_reg)

print("media di tutte le combinazioni: ", grid_search_reg.cv_results_["mean_test_score"])
print("parametri migliori: ", grid_search_reg.best_params_)
print("migliori risultati: ", grid_search_reg.best_score_)

media di tutte le combinazioni:  [0.33335289 0.33312651 0.33676205 0.33721214 0.33831162 0.33290586
 0.33542657 0.33731943 0.34358607 0.33909363 0.34004007 0.33918489
 0.3437947  0.34125776 0.34151645 0.34091742 0.34036769 0.33874636
 0.33576002 0.33579768 0.3404729  0.33333921 0.33711969 0.3385625
 0.33613645 0.3401517  0.33879545 0.34084376 0.3401988  0.34290273
 0.34453011 0.34088467 0.3286323  0.33448589 0.33304505 0.33368534
 0.32907884 0.32941862 0.33086076 0.33076123 0.34249846 0.33831987
 0.33876882 0.33876651 0.34058227 0.34450828 0.3436311  0.34420783
 0.2924576  0.30153628 0.30767684 0.30582518 0.32406876 0.31979282
 0.32110365 0.31877479 0.33323815 0.3343798  0.33655453 0.33718965
 0.34017432 0.34038721 0.34076698 0.34082161 0.2965028  0.30410176
 0.31157572 0.30079254 0.30905134 0.31946156 0.31747259 0.32072825
 0.33607218 0.33131398 0.33575891 0.33692599 0.33827486 0.34138251
 0.34179076 0.34167479 0.27522695 0.27988481 0.28226834 0.2786279
 0.27997188 0.29197452 0.289220

In [36]:
clf = RandomForestClassifier(**grid_search_clf.best_params_, n_jobs=-1, random_state=42)
reg = RandomForestRegressor(**grid_search_reg.best_params_, n_jobs=-1, random_state=42)
clf.fit(X_clf, y_clf)
reg.fit(X_reg, y_reg)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [37]:
y_pred_clf = clf.predict(X_test)
y_pred_reg = reg.predict(X_test)
y_pred2 = []
for (c, r) in zip(y_pred_clf, y_pred_reg):
    if c == False:
        y_pred2.append(0)
    else:
        y_pred2.append(r)

In [38]:
print("MAE: ", mean_absolute_error(y_pred1, y_test))
print("clf+reg: ", mean_absolute_error(y_pred2, y_test))

print("MSE: ", mean_squared_error(y_pred1, y_test))
print("clf+reg:", mean_squared_error(y_pred2, y_test))

# calcolo dell'accuracy

# converto le predizioni in numeri interi
y_pred1_int = [int(e) for e in y_pred1]
y_pred2_int = [int(e) for e in y_pred2]
# converto la lista di pandas in una lista normale
y_test_list = list(y_test)

conteggio1 = 0
conteggio2 = 0

# for che serve per contare quante volte il modello ha indovinato
for i in range(len(y_test_list)):
    if y_pred1_int[i] == y_test_list[i]:
        conteggio1 += 1
    if y_pred2_int[i] == y_test_list[i]:
        conteggio2 += 1

print("accuracy: ", conteggio1 / len(y_test_list))
print("clf+reg:", conteggio2 / len(y_test_list))

MAE:  19.14942486988467
clf+reg:  17.117712078286615
MSE:  889.7618268633428
clf+reg: 997.3561433704365
accuracy:  0.22712933753943218
clf+reg: 0.4794952681388013
